Absolutely! 🚀 Here's the **full working framework** for your **order management system** using **MCP servers with LangGraph**, including **parallel execution**, dynamic intent/city extraction via LLM, and agent orchestration.

---

# 📂 Project Structure

```
order_mgmt_framework/
│── main.py
│
├── config/
│   ├── __init__.py
│   ├── settings.py
│
├── mcp_servers/
│   ├── __init__.py
│   ├── weather_server.py
│   ├── pollution_server.py
│
├── mcp_clients/
│   ├── __init__.py
│   ├── weather_client.py
│   ├── pollution_client.py
│
├── tools/
│   ├── __init__.py
│   ├── weather_tools.py
│   ├── pollution_tools.py
│
├── agents/
│   ├── __init__.py
│   ├── agent_factory.py
│   ├── parent_agent.py
```

---

# 🔹 `config/settings.py`

```python
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DEFAULT_MODEL = "gpt-4o-mini"

AGENT_CONFIG = {
    "weather": {
        "tools": ["get_city_weather"],
        "mcp_servers": ["weather-mcp-1"]
    },
    "pollution": {
        "tools": ["get_city_pollution"],
        "mcp_servers": ["pollution-mcp-1"]
    },
    "parent": {
        "agents": ["weather", "pollution"],
        "llm_model": "gpt-4o-mini"
    }
}
```

---

# 🔹 `mcp_servers/weather_server.py`

```python
from fastmcp.server import FastMCP

WEATHER_DATA = {
    "Paris": "☁️ Cloudy, 22°C",
    "London": "🌧️ Rainy, 18°C",
    "Delhi": "☀️ Hot, 35°C"
}

mcp = FastMCP("weather-mcp-1")

@mcp.tool()
def get_city_weather(city: str):
    """Return weather info for a given city"""
    return {"content": WEATHER_DATA.get(city, "No weather data available")}

if __name__ == "__main__":
    print("Starting weather-mcp-1")
    mcp.run()
```

---

# 🔹 `mcp_servers/pollution_server.py`

```python
from fastmcp.server import FastMCP

POLLUTION_DATA = {
    "Delhi": "AQI 320 (Very Poor)",
    "Mumbai": "AQI 160 (Moderate)",
    "Paris": "AQI 70 (Good)"
}

mcp = FastMCP("pollution-mcp-1")

@mcp.tool()
def get_city_pollution(city: str):
    """Return pollution info for a given city"""
    return {"content": POLLUTION_DATA.get(city, "No pollution data available")}

if __name__ == "__main__":
    print("Starting pollution-mcp-1")
    mcp.run()
```

---

# 🔹 `mcp_clients/weather_client.py`

```python
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os

async def get_city_weather(city: str) -> str:
    """Client to call Weather MCP server"""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    server_path = os.path.join(current_dir, "..", "mcp_servers", "weather_server.py")

    server_params = StdioServerParameters(
        command="python",
        args=[server_path]
    )

    async with stdio_client(server_params) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            result = await session.call_tool("get_city_weather", {"city": city})
            return result.content[0].text if result.content else None
```

---

# 🔹 `mcp_clients/pollution_client.py`

```python
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os

async def get_city_pollution(city: str) -> str:
    """Client to call Pollution MCP server"""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    server_path = os.path.join(current_dir, "..", "mcp_servers", "pollution_server.py")

    server_params = StdioServerParameters(
        command="python",
        args=[server_path]
    )

    async with stdio_client(server_params) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            result = await session.call_tool("get_city_pollution", {"city": city})
            return result.content[0].text if result.content else None
```

---

# 🔹 `tools/weather_tools.py`

```python
import asyncio
from mcp_clients.weather_client import get_city_weather

class WeatherTools:
    @staticmethod
    def get_city_weather(city: str):
        return asyncio.run(get_city_weather(city))
```

---

# 🔹 `tools/pollution_tools.py`

```python
import asyncio
from mcp_clients.pollution_client import get_city_pollution

class PollutionTools:
    @staticmethod
    def get_city_pollution(city: str):
        return asyncio.run(get_city_pollution(city))
```

---

# 🔹 `agents/agent_factory.py`

```python
from tools.weather_tools import WeatherTools
from tools.pollution_tools import PollutionTools

class WeatherAgent:
    def run(self, city: str):
        return WeatherTools.get_city_weather(city)

class PollutionAgent:
    def run(self, city: str):
        return PollutionTools.get_city_pollution(city)

class AgentFactory:
    @staticmethod
    def create_agent(name: str):
        if name == "weather":
            return WeatherAgent()
        elif name == "pollution":
            return PollutionAgent()
        else:
            raise ValueError(f"Unknown agent: {name}")
```

---

# 🔹 `agents/parent_agent.py` (LangGraph + Parallel Execution)

```python
from langgraph.graph import StateGraph, END
from config.settings import AGENT_CONFIG
from agents.agent_factory import AgentFactory
from openai import OpenAI
import os
import json

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -------------------------
# State Schema
# -------------------------
class AgentState(dict):
    query: str
    intents: list  # ["weather", "pollution"]
    city: str
    results: dict
    final: str

# -------------------------
# Node: Classify Intent
# -------------------------
def classify_intent(state: AgentState) -> AgentState:
    prompt = f"""
    You are a classifier. Extract multiple intents (weather, pollution) and city from the query.
    User query: {state['query']}
    Respond in JSON strictly like: {{"intents": ["weather","pollution"], "city": "Delhi"}}
    """
    resp = client.chat.completions.create(
        model=AGENT_CONFIG["parent"]["llm_model"],
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        parsed = json.loads(resp.choices[0].message.content)
    except Exception:
        parsed = {"intents": [], "city": ""}
    state["intents"] = parsed.get("intents", [])
    state["city"] = parsed.get("city", "")
    state["results"] = {}
    return state

# -------------------------
# Node: Call Weather Agent
# -------------------------
def call_weather(state: AgentState) -> AgentState:
    if "weather" in state["intents"]:
        agent = AgentFactory.create_agent("weather")
        state["results"]["weather"] = agent.run(state["city"])
    return state

# -------------------------
# Node: Call Pollution Agent
# -------------------------
def call_pollution(state: AgentState) -> AgentState:
    if "pollution" in state["intents"]:
        agent = AgentFactory.create_agent("pollution")
        state["results"]["pollution"] = agent.run(state["city"])
    return state

# -------------------------
# Node: Merge Results
# -------------------------
def merge_results(state: AgentState) -> AgentState:
    if not state["results"]:
        state["final"] = "❌ No results found."
    else:
        parts = [f"{k.capitalize()}: {v}" for k, v in state["results"].items()]
        state["final"] = " | ".join(parts)
    return state

# -------------------------
# Build Parallel Graph
# -------------------------
workflow = StateGraph(AgentState)

workflow.add_node("classify", classify_intent)
workflow.add_node("weather", call_weather)
workflow.add_node("pollution", call_pollution)
workflow.add_node("merge", merge_results)

workflow.set_entry_point("classify")

# parallel execution
workflow.add_edge("classify", "weather")
workflow.add_edge("classify", "pollution")

# merge results
workflow.add_edge("weather", "merge")
workflow.add_edge("pollution", "merge")
workflow.add_edge("merge", END)

app = workflow.compile()
```

---

# 🔹 `main.py`

```python
from agents.parent_agent import app

if __name__ == "__main__":
    print("🤖 Order Management Framework with LangGraph started (parallel execution enabled)!")
    while True:
        query = input("\nYou: ")
        if query.lower() in ["exit", "quit"]:
            break
        result = app.invoke({"query": query})
        print("Bot:", result["final"])
```

---

### ✅ Features

1. **Dynamic LLM Intent Extraction** – extracts city and which agent(s) to call.
2. **Parallel Execution** – weather and pollution agents can run concurrently if requested.
3. **MCP Integration** – communicates with independent MCP servers via clients.
4. **Extensible** – add more agents/tools via `settings.py` and `agent_factory.py`.
5. **Final Output** – merges results from parallel execution.

---

Example:

```
You: Weather and pollution in Delhi
Bot: Weather: ☀️ Hot, 35°C | Pollution: AQI 320 (Very Poor)
```

---

If you want, I can **upgrade it further** so that **agents can be added dynamically just from settings.py**, and the graph automatically builds branches for them.

Do you want me to do that next?
